In [ ]:
# --- Environment setup: runs the same locally and on Google Colab ---
import os
import sys

try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Code comes from the cloned GitHub repo; run this in a cell first:
    #   !git clone https://github.com/muhtello/ironHack-project-deep_Learning.git
    CODE_ROOT = "/content/ironHack-project-deep_Learning"
    # Data (must contain extracted_animals/raw-img/) lives on Drive.
    # Adjust this to wherever you unzipped archive.zip in your Drive.
    DATA_ROOT = "/content/drive/MyDrive/animals10"
else:
    # Local machine: code and data share the project root.
    CODE_ROOT = os.getcwd()
    DATA_ROOT = os.getcwd()

# Make the .py modules importable regardless of where the notebook runs from.
sys.path.append(os.path.join(CODE_ROOT, "preprocessing", "label_mapping"))
sys.path.append(os.path.join(CODE_ROOT, "preprocessing", "data_loader"))
sys.path.append(os.path.join(CODE_ROOT, "model"))

In [ ]:
from label_mapping import build_labeled_dataset

# mapping the label; DATA_ROOT points at the folder holding extracted_animals/
df = build_labeled_dataset(project_root=DATA_ROOT)

In [ ]:
df.info()

In [ ]:
from data_loader import build_train_val_test_generators

# ImageDataGenerator resizes each image on load and rescales pixels to [0, 1]
# (normalization). Use 224x224 on GPU (Colab); drop to 128 for faster CPU runs.
# 70/15/15 train/val/test split.
image_size = (224, 224) if ON_COLAB else (128, 128)
train_generator, val_generator, test_generator = build_train_val_test_generators(
    df, project_root=DATA_ROOT, image_size=image_size
)

print(
    f"Train batches: {len(train_generator)}, "
    f"Val batches: {len(val_generator)}, "
    f"Test batches: {len(test_generator)}"
)
train_generator.class_indices

In [ ]:
from cnn_model import init_cnn_model

# input_shape and num_classes both read from the generator so the model always
# stays in sync with the data (image_shape is e.g. (224, 224, 3) on Colab).
model = init_cnn_model(
    input_shape=train_generator.image_shape,
    num_classes=len(train_generator.class_indices),
)
model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# Stop once val_loss stops improving for 3 epochs, and restore the weights
# from the best epoch so we don't keep an overfitted final state.
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=30,
    callbacks=[early_stopping],
)